# 01. Series universe and fixed-date classification

Goal: build the list of Kalshi series that enter the study, and pre-label each one as **fixed-date**, **deadline** or **review**.

The label must be decided **ex ante**, from the contract design only. Using fields that depend on the outcome (such as an early close) would reintroduce look-ahead bias.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import re
import pandas as pd
from pmcal import kalshi

pd.set_option("display.max_colwidth", 60)

## 1. All series with their volume

In [2]:
series = pd.DataFrame(kalshi.list_series(include_volume=True))
series["volume"] = pd.to_numeric(series["volume_fp"])
print(f"{len(series):,} series")
series.groupby("category")["volume"].agg(n_series="size", volume="sum").sort_values("volume", ascending=False)

14,288 series


,n_series,volume
category,,
Sports,3752,1.234065e+11
Crypto,274,3.105153e+10
Politics,2354,2.396065e+09
Elections,1780,1.933489e+09
Mentions,448,1.248037e+09
Commodities,83,1.178342e+09
Economics,809,1.174349e+09
Entertainment,2544,1.156212e+09
Climate and Weather,407,1.134308e+09


## 2. Exclusions and category families

- Series resolving in under 24 hours (`hourly`, `fifteen_min`) are excluded by protocol.
- Mention markets ("What will X say?") are excluded: they resolve Yes as soon as a word is said, which makes them deadline markets within the event.
- Kalshi categories are grouped into the six families of the protocol; the others are dropped.

In [3]:
FAMILIES = {
    "Sports": "sports",
    "Politics": "politics", "Elections": "politics",
    "Crypto": "crypto",
    "Economics": "economics", "Financials": "economics", "Commodities": "economics",
    "Climate and Weather": "weather",
    "Entertainment": "culture",
}
universe = series[~series["frequency"].isin(["hourly", "fifteen_min"])].copy()
universe["family"] = universe["category"].map(FAMILIES)
universe = universe.dropna(subset=["family"])
universe.groupby("family")["volume"].agg(n_series="size", volume="sum").sort_values("volume", ascending=False)

,n_series,volume
family,,
sports,3752,1.234065e+11
politics,4133,4.329550e+09
economics,1812,1.877248e+09
culture,2544,1.156212e+09
weather,393,1.086288e+09
crypto,235,3.043113e+08


## 3. Keep the top series of each family

Volume is extremely concentrated, and sports alone would dominate a volume-weighted sample. We therefore keep the **top 25 series per family**, which makes a manual review feasible and keeps every family represented.

In [4]:
TOP_N = 25
top = (universe.sort_values("volume", ascending=False)
       .groupby("family", group_keys=False).head(TOP_N)
       .reset_index(drop=True))
coverage = top.groupby("family")["volume"].sum() / universe.groupby("family")["volume"].sum()
print(f"{len(top)} series kept")
coverage.rename("share of family volume").to_frame().style.format("{:.0%}")

150 series kept


,share of family volume
family,
crypto,96%
culture,54%
economics,82%
politics,55%
sports,73%
weather,89%


## 4. Rule-based pre-labelling

- Scheduled recurring releases (`daily`, `weekly`, `monthly`, `quarterly`) are **fixed-date**.
- Single sports contests (game, match, fight, spread, total) are **fixed-date**.
- Titles suggesting "before a date" or "will X happen" are **deadline**.
- Everything else goes to **review**, to be labelled by hand.

In [5]:
SPORT_CONTEST = re.compile(r"\b(game|match|fight|spread|total)\b", re.I)
DEADLINE_HINT = re.compile(r"\b(before|by end|this year|shutdown|will .* happen|announce)\b", re.I)

def pre_label(row):
    if row["frequency"] in {"daily", "weekly", "monthly", "quarterly"}:
        return "fixed-date"
    if row["family"] == "sports" and SPORT_CONTEST.search(row["title"]):
        return "fixed-date"
    if DEADLINE_HINT.search(row["title"]):
        return "deadline"
    return "review"

top["label"] = top.apply(pre_label, axis=1)
pd.crosstab(top["family"], top["label"], margins=True)

label,deadline,fixed-date,review,All
family,,,,
crypto,6,9,10,25
culture,0,5,20,25
economics,0,14,11,25
politics,2,0,23,25
sports,0,19,6,25
weather,0,24,1,25
All,8,71,71,150


In [6]:
top.loc[top["label"] == "review", ["family", "ticker", "title", "frequency", "volume"]]

,family,ticker,title,frequency,volume
4,sports,KXATPCHALLENGERMATCH,Challenger ATP,custom,5.967851e+09
9,sports,KXPGATOUR,PGA Tour,custom,3.881064e+09
12,sports,KXWCADVANCE,World Cup Advance,custom,2.502023e+09
16,sports,KXMENWORLDCUP,Men's World Cup winner,custom,1.540162e+09
21,sports,KXWTACHALLENGERMATCH,Challenger WTA,custom,1.106753e+09
...,...,...,...,...,...
137,culture,KXDANCINGWITHTHESTARS,Who will win dancing with the stars?,one_off,7.749228e+06
142,crypto,KXCRYPTORETURNY,Crypto being positive,annual,1.878859e+06
144,crypto,KXDOGEMAX1,Dogecoin hitting $1,custom,1.470064e+06
145,crypto,KXXRPMAXY,XRP Max,annual,1.430142e+06


## 5. Manual review

Every series left in **review** is labelled by hand from its contract design, and the automatic rules are corrected where they fail (monthly "one touch" crypto series look scheduled but resolve as soon as the price touches the level).

A fourth label appears: **multi-stage**. These markets have a fixed end date, but outcomes are eliminated along the way (teams knocked out, candidates dropping out). A market that closes at elimination would be kept or dropped depending on its outcome, so these series are excluded from the main analysis.

In [7]:
MANUAL = {
    # fixed-date: resolution on a date known when the market opens
    "KXATPCHALLENGERMATCH": ("fixed-date", "single match"),
    "KXWTACHALLENGERMATCH": ("fixed-date", "single match"),
    "KXWCSCORE": ("fixed-date", "single match score"),
    "KXFEDDECISION": ("fixed-date", "scheduled FOMC meeting"),
    "KXFED": ("fixed-date", "scheduled FOMC meeting"),
    "KXGDP": ("fixed-date", "scheduled data release"),
    "PRES": ("fixed-date", "election day"),
    "POPVOTE": ("fixed-date", "election day"),
    "POPVOTEMOV": ("fixed-date", "election day"),
    "POPVOTEMOVSMALL": ("fixed-date", "election day"),
    "POPVOTEMOVSMALLER": ("fixed-date", "election day"),
    "ECMOV": ("fixed-date", "election day"),
    "PRESPARTYFULL": ("fixed-date", "election day"),
    "KXPRESPERSON": ("fixed-date", "election day"),
    "CONTROLH": ("fixed-date", "election day"),
    "POWER": ("fixed-date", "election day"),
    "SENATEAZ": ("fixed-date", "election day"),
    "KXSENATEMID": ("fixed-date", "election day"),
    "KXGOVCA": ("fixed-date", "election day"),
    "KXMAYORLA": ("fixed-date", "election day"),
    "KXMAYORNYCPARTY": ("fixed-date", "election day"),
    "KXVOTEPRIMARY": ("fixed-date", "primary election day"),
    "KXGOVFLNOMR": ("fixed-date", "primary election day"),
    "KXOSCARPIC": ("fixed-date", "ceremony date"),
    "KXOSCARACTO": ("fixed-date", "ceremony date"),
    "KXOSCARACTR": ("fixed-date", "ceremony date"),
    "KXOSCARSUPACTO": ("fixed-date", "ceremony date"),
    "KXOSCARNOMPIC": ("fixed-date", "nominations announcement date"),
    "KXGRAMSOTY": ("fixed-date", "ceremony date"),
    "KXFIRSTSUPERBOWLSONG": ("fixed-date", "halftime show date"),
    "KXSBGUESTS": ("fixed-date", "halftime show date"),
    "KXRT": ("fixed-date", "score read on a set date"),
    "KXTIME": ("fixed-date", "scheduled announcement"),
    "KXTOPARTIST": ("fixed-date", "year-end ranking"),
    "KXRANKLISTGOOGLESEARCH": ("fixed-date", "year-end ranking"),
    "KXLIUSAELIMINATION": ("fixed-date", "single episode"),
    "KXLIUSAELIMINATIONW": ("fixed-date", "single episode"),
    "KXBTCY": ("fixed-date", "end-of-year price"),
    "KXETHY": ("fixed-date", "end-of-year price"),
    "KXCRYPTORETURNY": ("fixed-date", "end-of-year return"),
    "KXINXY": ("fixed-date", "end-of-year close"),
    "KXNASDAQ100Y": ("fixed-date", "end-of-year close"),
    # deadline: can resolve Yes as soon as something happens
    "KXFEDCHAIRNOM": ("deadline", "resolves when the nomination is made"),
    "KXGOVTSHUTDOWN": ("deadline", "resolves when a shutdown starts"),
    "KXKHAMENEIOUT": ("deadline", "resolves when it happens"),
    "KXHORMUZNORM": ("deadline", "resolves when it happens"),
    "KXCITRINI": ("deadline", "resolves when it happens"),
    "KXPERFORMSUPERBOWLB": ("deadline", "resolves when the performer is announced"),
    "KXRATECUT": ("deadline", "resolves at the first cut"),
    "KXRATECUTCOUNT": ("deadline", "thresholds hit before year end"),
    "KXLAYOFFSYINFO": ("deadline", "thresholds hit before year end"),
    "KXGOVTCUTS": ("deadline", "thresholds hit before year end"),
    "KXALBUMSALES": ("deadline", "thresholds hit before the end date"),
    "KXTOPALBUMRECORD": ("deadline", "record can be broken early"),
    "KXWTIMAX": ("deadline", "touch market"),
    "KXBTCMAX100": ("deadline", "touch market"),
    "KXBTCMAX125": ("deadline", "touch market"),
    "KXBTCMAX150": ("deadline", "touch market"),
    "KXBTC2026200": ("deadline", "touch market"),
    "KXBTC2026250": ("deadline", "touch market"),
    "KXDOGEMAX1": ("deadline", "touch market"),
    "KXXRPMAXY": ("deadline", "touch market"),
    "KXSNOWSTORM": ("deadline", "unclear contract, excluded to be safe"),
    # multi-stage: fixed end date, but outcomes are eliminated along the way
    "KXMENWORLDCUP": ("multi-stage", "teams eliminated before the final"),
    "KXWCADVANCE": ("multi-stage", "can be decided before the last group game"),
    "KXPGATOUR": ("multi-stage", "players cut during the tournament"),
    "KXPRESNOMD": ("multi-stage", "candidates drop out during the primaries"),
    "KXPRESNOMR": ("multi-stage", "candidates drop out during the primaries"),
    "KXSURVIVOR": ("multi-stage", "contestants eliminated during the season"),
    "KXDANCINGWITHTHESTARS": ("multi-stage", "contestants eliminated during the season"),
    "KXLIUSACOUPLE": ("multi-stage", "couples eliminated during the season"),
    # corrections of the automatic rules: monthly "one touch" crypto series are touch markets
    "KXBTCMAXMON": ("deadline", "touch market"),
    "KXBTCMINMON": ("deadline", "touch market"),
    "KXETHMAXMON": ("deadline", "touch market"),
    "KXETHMINMON": ("deadline", "touch market"),
    "KXXRPMAXMON": ("deadline", "touch market"),
    "KXSOLMAXMON": ("deadline", "touch market"),
    "KXBNBMINMON": ("deadline", "touch market"),
    "KXBTCMAXM": ("deadline", "touch market"),
}

top["reason"] = top["label"].map({"fixed-date": "automatic rule", "deadline": "automatic rule"})
for ticker, (label, reason) in MANUAL.items():
    top.loc[top["ticker"] == ticker, ["label", "reason"]] = [label, reason]
assert (top["label"] != "review").all(), "some series are still unlabelled"
pd.crosstab(top["family"], top["label"], margins=True)

label,deadline,fixed-date,multi-stage,All
family,,,,
crypto,21,4,0,25
culture,3,19,3,25
economics,6,19,0,25
politics,6,17,2,25
sports,0,22,3,25
weather,1,24,0,25
All,37,105,8,150


## 6. Export

The file is committed to the repo: the labels are part of the protocol and can be audited.

In [8]:
out = ROOT / "data" / "series_universe.csv"
cols = ["family", "category", "ticker", "title", "frequency", "fee_type", "fee_multiplier", "volume", "label", "reason"]
top[cols].to_csv(out, index=False)
print(f"Saved {len(top)} series to {out.relative_to(ROOT)}")

Saved 150 series to data/series_universe.csv
